<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 6 — Fine-tuning de BERT</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Un backbone, tres cabezas — con datasets reales y demo sobre su corpus</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Entorno: Google Colab con GPU T4 (free tier).** Entorno de ejecución → Cambiar tipo → **GPU**. Los tres modelos (BETO 110M, Sentence-BERT) caben de sobra en los 15 GB de la T4 usando `fp16`. Esta versión entrena con **datasets reales de HuggingFace** (no con el corpus de juguete): `tweet_sentiment_multilingual` para clasificación y `conll2002` para NER. El corpus chiapaneco se usa solo para la **inferencia final** de cada parte. Las celdas de *liberar memoria* permiten correr A → B → C en una sola sesión sin saturar la VRAM.


## Objetivo

Tres partes, el mismo BERT preentrenado en español como base. **A)** Afinar un encoder de oraciones (Sentence-BERT) con sus pares del Lab 3. **B)** Clasificar **sentimiento** con un dataset real en español. **C)** **NER** con CoNLL-2002. Cada parte cierra **usando el modelo afinado** sobre el corpus chiapaneco, y libera memoria antes de la siguiente.


## 0 · Setup, GPU y utilidades

In [28]:
# Instalación (Colab). Reiniciar el entorno si lo pide tras instalar.
!pip install -q transformers sentence-transformers datasets seqeval accelerate
!pip install -q datasets==2.21.0

import gc, math, json
import numpy as np
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — activen el runtime de GPU')

def liberar_memoria():
    """Libera RAM/VRAM. Llamar tras borrar (del) las variables del entrenamiento previo."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f'VRAM en uso: {torch.cuda.memory_allocated()/1e9:.2f} GB')

GPU: Tesla T4


In [29]:
# El corpus chiapaneco (del Lab 1) se usa SOLO para la inferencia final de cada parte.
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}
ids = [d['id'] for d in corpus]; titulos = {d['id']: d['titulo'] for d in corpus}
print(len(corpus), 'documentos del corpus cargados (para inferencia).')

14 documentos del corpus cargados (para inferencia).


---
## Parte A · Embeddings con Sentence-BERT (datos: sus qrels del Lab 3)

> **Por qué aquí sí usamos el corpus.** El fine-tuning de embeddings necesita pares *de dominio* (consulta ↔ documento relevante). Esos pares salen de **sus** qrels del Lab 3: es el cierre del arco de la unidad, donde su juicio de relevancia se vuelve señal de entrenamiento. **Advertencia metodológica:** con ~5 consultas esto es una *demostración del método*, no un experimento — la mejora puede ser chica o ruidosa. Lo evaluable es el pipeline correcto, no el número.


**A.1** Línea base: carguen un Sentence-BERT en español y midan su buscador **sin afinar** con su nDCG del Lab 3.

In [30]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
modelo = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')
# TODO: reusen sus qrels del Lab 3; funciones emb_corpus, buscar (coseno) y ndcg_medio.
# TODO: midan el nDCG del modelo SIN afinar (linea base a superar).
qrels = {
    'sequia y cultivos':                {'d04': 3, 'd02': 2},
    'problemas de agua':                {'d02': 3, 'd13': 3, 'd01': 1},
    'cafe de chiapas':                  {'d03': 3},
    'turismo en el canon del sumidero': {'d05': 3},
    'tecnologia en la universidad':     {'d07': 3, 'd14': 2},
}

def ndcg_at_k(ranking, qid, k=5):
    def rel(d): return qrels[qid].get(d, 0)
    dcg  = sum(rel(d) / math.log2(i+1) for i, d in enumerate(ranking[:k], 1))
    idcg = sum(g / math.log2(i+1)
               for i, g in enumerate(sorted(qrels[qid].values(), reverse=True)[:k], 1))
    return dcg / idcg if idcg else 0.0

textos_corpus = [crudo[id_] for id_ in ids]
EMB1 = modelo.encode(textos_corpus, convert_to_numpy=True, show_progress_bar=True)

def buscar_sbert(consulta, emb_corpus, k=None):
    k = k or len(ids)
    q_vec = modelo.encode([consulta], convert_to_numpy=True)[0]
    sims  = [(ids[i], float(np.dot(q_vec, emb_corpus[i]) /
              (np.linalg.norm(q_vec) * np.linalg.norm(emb_corpus[i]) + 1e-9)))
             for i in range(len(ids))]
    return sorted(sims, key=lambda x: x[1], reverse=True)[:k]

def ndcg_medio(emb_corpus, k=5):
    total = 0.0
    for qid in qrels:
        ranking = [doc_id for doc_id, _ in buscar_sbert(qid, emb_corpus, k=len(ids))]
        total  += ndcg_at_k(ranking, qid, k)
    return total / len(qrels)

ndcg_base = ndcg_medio(EMB1)
print(f'nDCG@5 SBERT sin afinar: {ndcg_base:.4f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

nDCG@5 SBERT sin afinar: 0.8000


**A.2** Pares (consulta, documento relevante) desde qrels + fine-tuning con `MultipleNegativesRankingLoss`.

In [31]:
# TODO: InputExample(texts=[consulta, documento]) para positivos (g>=2); DataLoader;
#       MultipleNegativesRankingLoss; modelo.fit(epochs=2, warmup_steps=...). Recalcular nDCG.
pares = [
    InputExample(texts=[qid, crudo[doc_id]])
    for qid, docs in qrels.items()
    for doc_id, grado in docs.items()
    if grado >= 2
]
print(f'Pares de entrenamiento: {len(pares)}')

train_loader = DataLoader(pares, shuffle=True, batch_size=4)
loss_fn      = losses.MultipleNegativesRankingLoss(model=modelo)

modelo.fit(
    train_objectives=[(train_loader, loss_fn)],
    epochs=2,
    warmup_steps=2,
    show_progress_bar=True,
)

EMB2         = modelo.encode(textos_corpus, convert_to_numpy=True, show_progress_bar=True)
ndcg_afinado = ndcg_medio(EMB2)

print(f'nDCG@5 SBERT sin afinar: {ndcg_base:.4f}')
print(f'nDCG@5 SBERT afinado:    {ndcg_afinado:.4f}')
print(f'Delta:                   {ndcg_afinado - ndcg_base:+.4f}')

Pares de entrenamiento: 8


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

nDCG@5 SBERT sin afinar: 0.8000
nDCG@5 SBERT afinado:    0.8827
Delta:                   +0.0827


_Reporten los tres nDCG (FastText, SBERT base, SBERT afinado) y comenten:_como se muestran el afinado no tuvo mucho cambio y no supero a fastext debido a que faltan mas datos para mejorarlo  y que sea mas efectivo

**A.3 · Uso del modelo afinado.** Busquen con una consulta nueva usando el encoder ya entrenado.

In [32]:
# TODO: con el modelo afinado y EMB1, busquen una consulta nueva de su eleccion (k=5).
consulta_nueva = 'desastres naturales en Chiapas'
print(f'Búsqueda: "{consulta_nueva}"')
for doc_id, score in buscar_sbert(consulta_nueva, EMB2, k=5):
    print(f'  {score:.4f}  {doc_id}  {titulos[doc_id]}')

Búsqueda: "desastres naturales en Chiapas"
  0.4679  d06  Sismo de magnitud 5.1 frente a las costas
  0.4337  d01  Lluvias provocan inundaciones en Tuxtla
  0.4016  d11  Alertan por casos de dengue
  0.3713  d07  UPCh inaugura laboratorio de IA
  0.3603  d10  Avanza obra de infraestructura carretera


**Liberar memoria** antes del siguiente entrenamiento (clave en T4).

In [33]:
# Borren las variables del modelo de esta parte y liberen VRAM:
del modelo, EMB1, EMB2
liberar_memoria()

VRAM en uso: 0.46 GB


---
## Parte B · Clasificación de sentimiento (dataset real en español)

Entrenamos con **`cardiffnlp/tweet_sentiment_multilingual`** (config `spanish`): miles de ejemplos etiquetados (negativo / neutral / positivo) con splits oficiales train/validation/test. *Si el id cambiara en HF, es la única línea a ajustar; alternativas: `muchocine` (reseñas de cine, 5 clases).*


**B.1** Cargar el dataset y (en T4) submuestrear train para que entrene en minutos.

In [34]:
from datasets import load_dataset
ds = load_dataset('cardiffnlp/tweet_sentiment_multilingual', 'spanish')
# TODO: obtener la lista de clases (features['label'].names) y los mapeos id2lab/lab2id.
# TODO: submuestrear ds['train'] (~2000) para T4 y tomar ds['test'].
clases = ds['train'].features['label'].names
id2lab = {i: c for i, c in enumerate(clases)}
lab2id = {c: i for i, c in id2lab.items()}
print('Clases:', clases)
print('Train:', len(ds['train']), '| Test:', len(ds['test']))

ds_tr = ds['train'].shuffle(seed=42).select(range(min(2000, len(ds['train']))))
ds_te = ds['test']
print(f'Train submuestreado: {len(ds_tr)} | Test: {len(ds_te)}')

Clases: ['negative', 'neutral', 'positive']
Train: 1839 | Test: 870
Train submuestreado: 1839 | Test: 870


**B.2** Tokenizar con el tokenizer de BETO.

In [35]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)
# TODO: tokenizar (truncation + padding, max_length=128) train y test -> ds_tr, ds_te.
def tokenizar(batch):
    return tok(batch['text'], truncation=True, padding='max_length', max_length=128)

ds_tr = ds_tr.map(tokenizar, batched=True)
ds_te = ds_te.map(tokenizar, batched=True)
ds_tr.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
ds_te.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print('Tokenización completa.')

Tokenización completa.


**B.3** Fine-tuning con `Trainer` (LR 2e-5, pocas épocas, `fp16` para la T4).

In [36]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
# TODO: AutoModelForSequenceClassification (num_labels, id2label, label2id);
#       compute_metrics (accuracy + f1_macro); TrainingArguments (lr 2e-5, fp16=True en T4);
#       Trainer; entrenar y evaluar.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=len(clases), id2label=id2lab, label2id=lab2id,
)

args = TrainingArguments(
    output_dir='./sentiment_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_tr, eval_dataset=ds_te,
    compute_metrics=compute_metrics,
)
trainer.train()
print(trainer.evaluate())

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.929467,0.823019,0.651724,0.647114
2,0.664352,0.744644,0.678161,0.678841
3,0.479155,0.768860,0.682759,0.683977


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.479155,0.768860,3,0.682759,0.683977


{'eval_loss': 0.7688602805137634, 'eval_accuracy': 0.6827586206896552, 'eval_f1_macro': 0.6839769850990778}


**B.4** Análisis de errores: matriz de confusión y reporte por clase.

In [37]:
# TODO: classification_report + confusion_matrix sobre el test. ¿Que clase se confunde mas?
from sklearn.metrics import classification_report, confusion_matrix

preds_out = trainer.predict(ds_te)
preds     = np.argmax(preds_out.predictions, axis=-1)
labels    = preds_out.label_ids

print(classification_report(labels, preds, target_names=clases))

cm = confusion_matrix(labels, preds)
print('\nMatriz de confusión:')
print(f'{"":>12}', '  '.join(f'{c:>10}' for c in clases))
for i, fila in enumerate(cm):
    print(f'{clases[i]:>12}', '  '.join(f'{v:>10}' for v in fila))

              precision    recall  f1-score   support

    negative       0.74      0.70      0.72       290
     neutral       0.58      0.61      0.60       290
    positive       0.74      0.73      0.74       290

    accuracy                           0.68       870
   macro avg       0.69      0.68      0.68       870
weighted avg       0.69      0.68      0.68       870


Matriz de confusión:
               negative     neutral    positive
    negative        203          64          23
     neutral         61         178          51
    positive         12          65         213


_¿Qué clase es la más difícil? ¿Accuracy o F1-macro es mejor criterio aquí? ¿Por qué?_neutral ya que no hay un objetivo o una forma de que el modelo identifique  mas facilmente, lo que lo hace dudar,  f1-macro es mejor debido a que calcula por separado el resultado y despues promedia por lo que si hay un error se nota o de encuentra a diferencia de accuracy que no se puede detectar

**B.5 · Uso del modelo afinado** — transferencia al corpus chiapaneco. El modelo se entrenó con tuits; veamos qué predice sobre noticias.

In [38]:
from transformers import pipeline
# TODO: crear un pipeline('text-classification') con su modelo afinado y aplicarlo a
#       3-4 frases propias + algunas del corpus (crudo[...]). Comenten el domain shift.
clf = pipeline(
    'text-classification', model=model, tokenizer=tok,
    device=0 if torch.cuda.is_available() else -1,
)

textos_prueba = [
    'El café de Chiapas rompe récords históricos de exportación.',
    'La sequía destruyó todos los cultivos este año.',
    'El sismo no causó daños ni víctimas.',
] + [crudo[id_] for id_ in ['d01', 'd02', 'd11', 'd14']]

for texto in textos_prueba:
    r = clf(texto[:512])[0]
    print(f'[{r["label"]:>8}  {r["score"]:.2f}]  {texto[:65]}...')

[positive  0.81]  El café de Chiapas rompe récords históricos de exportación....
[negative  0.59]  La sequía destruyó todos los cultivos este año....
[ neutral  0.53]  El sismo no causó daños ni víctimas....
[ neutral  0.54]  Las  fuertes lluvias   provocaron inundaciones en varias colonias...
[negative  0.83]  La crisis hidrica se agrava: el desabasto del liquido vital afect...
[negative  0.66]  La Secretaria de Salud alerto por un repunte de casos de dengue e...
[ neutral  0.53]  Estudiantes de ingenieria de Tuxtla ganaron el primer lugar en un...


**Liberar memoria** antes de la Parte C.

In [39]:
del model, trainer, clf           # y el trainer/pipeline si los nombraron distinto
liberar_memoria()

VRAM en uso: 0.46 GB


---
## Parte C · NER con CoNLL-2002 (español)

Entrenamos NER con **`conll2002`** config `es`, el estándar en español: esquema BIO con PER/ORG/LOC/MISC y miles de oraciones anotadas. *Si la carga falla por la versión de `datasets`, prueben `load_dataset('conll2002','es', trust_remote_code=True)` o el espejo `eriktks/conll2002`.*


**C.1** Cargar el dataset y leer el esquema de etiquetas desde sus features.

In [40]:
from datasets import load_dataset
conll = load_dataset('eriktks/conll2002', 'es', trust_remote_code=True)
# TODO: obtener 'etiquetas' (features['ner_tags'].feature.names) y los mapeos id2lab/lab2id.
etiquetas  = conll['train'].features['ner_tags'].feature.names
id2lab_ner = {i: e for i, e in enumerate(etiquetas)}
lab2id_ner = {e: i for i, e in id2lab_ner.items()}
print('Etiquetas:', etiquetas)
print('Train:', len(conll['train']), '| Test:', len(conll['test']))

Etiquetas: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train: 8324 | Test: 1518


**C.2 — el corazón del lab.** Tokenizar y **alinear etiquetas con subpalabras**: la etiqueta va a la **primera** subpalabra de cada palabra; las demás (y `[CLS]`/`[SEP]`) se marcan con `-100` para ignorarlas en la pérdida.

In [41]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)
def tokeniza_y_alinea(batch):
    # TODO: tok(..., is_split_into_words=True); por cada ejemplo usar enc.word_ids(batch_index=i)
    #       para poner la etiqueta SOLO en la primera subpalabra y -100 en las demas y en [CLS]/[SEP].
    enc = tok(
        batch['tokens'],
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=128,
    )
    etqs_todas = []
    for i, tags in enumerate(batch['ner_tags']):
        word_ids  = enc.word_ids(batch_index=i)
        prev_word = None
        etqs_ej   = []
        for wid in word_ids:
            if wid is None:
                etqs_ej.append(-100)
            elif wid != prev_word:
                etqs_ej.append(tags[wid])
            else:
                etqs_ej.append(-100)
            prev_word = wid
        etqs_todas.append(etqs_ej)
    enc['labels'] = etqs_todas
    return enc
# conll_tok = conll.map(tokeniza_y_alinea, batched=True, remove_columns=conll['train'].column_names)
cols_orig = conll['train'].column_names
conll_tok = conll.map(tokeniza_y_alinea, batched=True, remove_columns=cols_orig)
conll_tok.set_format(type='torch')
print('Alineación completa.')

ejemplo = tok(['Tuxtla', 'Gutiérrez'], is_split_into_words=True, return_tensors='pt')
print('\nSubpalabras de "Tuxtla Gutiérrez":', tok.convert_ids_to_tokens(ejemplo['input_ids'][0]))
print('word_ids:', ejemplo.word_ids(batch_index=0))
print('→ Solo la primera subpalabra de cada palabra recibe la etiqueta; las demás = -100')

Map:   0%|          | 0/8324 [00:00<?, ? examples/s]

Map:   0%|          | 0/1916 [00:00<?, ? examples/s]

Map:   0%|          | 0/1518 [00:00<?, ? examples/s]

Alineación completa.

Subpalabras de "Tuxtla Gutiérrez": ['[CLS]', 'Tu', '##x', '##t', '##la', 'Gutiérrez', '[SEP]']
word_ids: [None, 0, 0, 0, 0, 1, None]
→ Solo la primera subpalabra de cada palabra recibe la etiqueta; las demás = -100


**C.3** Fine-tuning con `AutoModelForTokenClassification` (`fp16`, T4).

In [42]:
from transformers import (AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification)
# TODO: AutoModelForTokenClassification (num_labels, id2label, label2id);
#       DataCollatorForTokenClassification; TrainingArguments (fp16 en T4); Trainer; entrenar.
model_ner = AutoModelForTokenClassification.from_pretrained(
    CKPT,
    num_labels=len(etiquetas),
    id2label=id2lab_ner,
    label2id=lab2id_ner,
)

collator = DataCollatorForTokenClassification(tok)

args_ner = TrainingArguments(
    output_dir='./ner_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    eval_strategy='epoch',
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
)

trainer_ner = Trainer(
    model=model_ner, args=args_ner,
    train_dataset=conll_tok['train'],
    eval_dataset=conll_tok['validation'],
    data_collator=collator,
)
trainer_ner.train()
print('Entrenamiento NER completo.')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.050788,0.086461
2,0.037747,0.078050
3,0.022107,0.083130


Entrenamiento NER completo.


**C.4** Evaluación con **seqeval** (a nivel de entidad, no de token).

In [43]:
from seqeval.metrics import classification_report as seq_report
# TODO: predecir; reconstruir secuencias ignorando -100; evaluar con seqeval (entidad).
#       Expliquen por que NO se usa accuracy por token.
preds_out = trainer_ner.predict(conll_tok['test'])
preds_ids = np.argmax(preds_out.predictions, axis=-1)
labels_ids = preds_out.label_ids

y_true, y_pred = [], []
for pred_seq, label_seq in zip(preds_ids, labels_ids):
    t_seq, p_seq = [], []
    for p, l in zip(pred_seq, label_seq):
        if l != -100:
            t_seq.append(id2lab_ner[l])
            p_seq.append(id2lab_ner[p])
    y_true.append(t_seq)
    y_pred.append(p_seq)

print(seq_report(y_true, y_pred))

              precision    recall  f1-score   support

         LOC       0.88      0.87      0.87      1070
        MISC       0.66      0.67      0.67       340
         ORG       0.85      0.90      0.87      1396
         PER       0.96      0.97      0.97       719

   micro avg       0.86      0.88      0.87      3525
   macro avg       0.84      0.85      0.85      3525
weighted avg       0.86      0.88      0.87      3525



_¿Por qué seqeval y no accuracy por token? ¿Qué tipo de entidad cuesta más?_debido a que seqeval obliga a detectar la entiodad completa  pero si hay un error la entidad completa falla como error por completo, asi es mas preciso ya que o le atinas o lo descarta completo y no a medias,  la que mas falla es misc ya que no encaja, no tiene una forma de destacarlos o una caracteristica que los diferencie, es como meter todo lo que no encaja en un monton

**C.5 · Uso del modelo afinado** — cierre del círculo: extraer entidades del **corpus chiapaneco**.

In [44]:
from transformers import pipeline
# TODO: pipeline('ner', aggregation_strategy='simple') con su modelo; aplicarlo a 3 documentos
#       del corpus (crudo[...]) e imprimir las entidades detectadas con su tipo y score.
ner_pipe = pipeline(
    'ner', model=model_ner, tokenizer=tok,
    aggregation_strategy='simple',
    device=0 if torch.cuda.is_available() else -1,
)

for doc_id in ['d01', 'd05', 'd07']:
    print(f'\n{doc_id} — {titulos[doc_id]}')
    entidades = ner_pipe(crudo[doc_id][:512])
    for e in entidades:
        print(f"  [{e['entity_group']:6}  {e['score']:.2f}]  {e['word']}")


d01 — Lluvias provocan inundaciones en Tuxtla
  [LOC     0.91]  Tuxtla
  [PER     0.96]  Gutierrez 😟
  [ORG     0.92]  Proteccion Civil
  [MISC    0.65]  h
  [MISC    0.57]  ##ttps : /
  [ORG     0.63]  chiapasparalelo

d05 — Turismo crece en el Canon del Sumidero
  [MISC    0.98]  Canon del Sumidero
  [LOC     0.98]  Chiapas

d07 — UPCh inaugura laboratorio de IA
  [ORG     0.99]  Universidad Politecnica de Chiapas
  [MISC    0.95]  G
  [MISC    0.69]  h
  [MISC    0.67]  ##ttps : / /
  [ORG     0.43]  up
  [MISC    0.47]  ##chi
  [ORG     0.64]  ##apas


**Liberar memoria** al terminar.

In [45]:
del model_ner, trainer_ner, ner_pipe
liberar_memoria()

VRAM en uso: 0.46 GB


---
## Síntesis

Las tres partes usaron **el mismo BERT preentrenado** y solo cambiaron la cabeza y los datos: cabeza siamesa con sus qrels (A), `[CLS]` + lineal con un dataset de sentimiento real (B), y una etiqueta por token con CoNLL-2002 (C). Cada modelo afinado se aplicó **sobre su propio corpus**, y entre entrenamientos se liberó memoria para sostener la sesión en una T4. Ese paradigma —preentrenar una vez, adaptar barato— es la base de los sistemas RAG de la Unidad 3.


## Entregables — Lab 6
- [ ] **A:** fine-tuning de Sentence-BERT + los tres nDCG + búsqueda con el modelo afinado.
- [ ] **B:** clasificación con dataset real + matriz de confusión + inferencia sobre el corpus.
- [ ] **C:** NER con CoNLL-2002 + alineación de subpalabras + seqeval + extracción sobre el corpus.
- [ ] Celdas de liberación de memoria ejecutadas entre partes.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
